In [ ]:
import json
import cv2
import numpy as np

def run_botsort(model_path, video_path, tracker_path, save_video=False, output_dir=None, stride=1):

    print("Running BotSort tracking...")

    results = model.track(
        source=video_path,
        tracker=tracker_path,
        stream=True,
        persist=True,
        conf=0.35,
        device=0,
        vid_stride=stride,
        verbose=False
    )

    tracking_results = []
    tracking_data = []
    video_writer = None

    for frame_i, result in enumerate(results):

        frame_record = {
            "frame": frame_i,
            "robots": [],
            "reef": []
        }

        # ----------------------------
        # SAFELY EXTRACT DATA
        # ----------------------------
        if result.boxes is not None and result.boxes.xyxy is not None:

            boxes = result.boxes.xyxy.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)

            # ✅ FIX: properly define track_ids
            if result.boxes.id is not None:
                track_ids = result.boxes.id.cpu().numpy().astype(int)
            else:
                track_ids = [None] * len(boxes)

        else:
            boxes, classes, track_ids = [], [], []

        # ----------------------------
        # STORE TRACKING DATA (NEW)
        # ----------------------------
        for box, track_id, cls in zip(boxes, track_ids, classes):

            x1, y1, x2, y2 = box

            if track_id is None:
                continue  # skip untracked objects

            if int(cls) == ROBOT_CLASS_ID:
                frame_record["robots"].append({
                    "id": int(track_id),
                    "bbox": [float(x1), float(y1), float(x2), float(y2)]
                })

            elif int(cls) == REEF_CLASS_ID:
                frame_record["reef"].append({
                    "bbox": [float(x1), float(y1), float(x2), float(y2)]
                })

        # ALWAYS append frame data (even if empty)
        tracking_data.append(frame_record)

        # ----------------------------
        # EXISTING TRACKING LOGIC
        # ----------------------------
        frame = result.plot()

        # Only proceed if IDs exist
        if len(track_ids) == 0:
            continue

        # Filter robot indices
        robot_indices = [i for i, c in enumerate(classes) if c == ROBOT_CLASS_ID]

        for i in robot_indices:

            if track_ids[i] is None:
                continue

            x1, y1, x2, y2 = boxes[i]
            track_id = track_ids[i]

            tracking_results.append({
                "frame": frame_i * stride,
                "track_id": int(track_id),
                "box": [float(x1), float(y1), float(x2), float(y2)],
                "crop": frame[int(y1):int(y2), int(x1):int(x2)].copy()
            })

            # DRAW ID
            cv2.putText(
                frame,
                f"ID {track_id}",
                (int(x1), int(y1) - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

        # ----------------------------
        # SAVE VIDEO
        # ----------------------------
        if save_video:
            if video_writer is None:
                h, w = frame.shape[:2]
                output_path = output_dir / "botsort_with_ids.mp4"

                video_writer = cv2.VideoWriter(
                    str(output_path),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    30,
                    (w, h)
                )

            video_writer.write(frame)

    if video_writer:
        video_writer.release()

    # ----------------------------
    # SAVE TRACKING JSON
    # ----------------------------
    with open("tracking.json", "w") as f:
        json.dump(tracking_data, f)

    print("Tracking data saved to tracking.json")
    print("Tracking complete.")

    return tracking_results